<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/IIRSI/Project/Step_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Этап 11. Celery, кэш и фоновые задачи — подробная методичка

Эта методичка продолжает Этапы 1–10. Предполагается, что у вас:
- `db`, `redis`, `qdrant`, `ollama`, `app` в `healthy`;
- `/health` — 200, все `services.*.healthy: true`;
- `perform_translation`, `/translate`, `/translate/multi`, `/translate/batch`, `/translate/ensemble` — 200;
- HITL работает: `/hitl/uncertain`, `/hitl/uncertain/{id}/confirm`, `/hitl/uncertain/{id}/reject`;
- **в `backend/hitl/active_learning.py` и `backend/hitl/feedback.py` уже стоят импорты `from backend.worker.tasks import index_translation_memory`** (с Этапа 10) — но модуля `backend/worker/tasks.py` **нет**, поэтому `confirm_correction`/`save_correction` при `inserted > 0` падают с `ImportError`. На этом этапе создаём `tasks.py` — связка оживает.

Если чего-то нет — вернитесь к Этапу 10.

**Цель этапа:** Celery worker + beat, 6 фоновых задач (переиндексация, batch-перевод, eval, очистка кэша, bulk-import), persistent event loop на процесс, корректный shutdown.

**Недель:** 1.

**Что вы получите в конце этапа:**

- `backend/worker/celery_app.py` — `celery_app`, конфиг, 2 shutdown-сигнала.
- `backend/worker/tasks.py` — 6 задач + persistent loop (`_get_worker_loop`, `_run_async`, `close_worker_loop`), константа `_MAX_CONCURRENT_TRANSLATIONS = 4`.
- `infrastructure/docker-compose.yml` — сервисы `worker` и `beat`.
- Работающая связка HITL → Celery (индексация после confirm/save).
- Тесты `tests/unit/test_tasks.py` с `task_always_eager=True`.

**Критерии готовности:**

- [ ] `docker compose ps` — `worker` и `beat` в `running`.
- [ ] `docker compose logs worker` содержит `celery@... ready` (или похожий старт), нет traceback.
- [ ] `confirm_correction` (Этап 10) после успешного commit вызывает `index_translation_memory.delay()`, задача видна в логах worker'а.
- [ ] `add_tm_bulk` при `added > 0` вызывает `index_translation_memory.delay()`.
- [ ] При остановке worker (`docker compose stop worker`) лог содержит `Worker event loop closed` и НЕ содержит `RuntimeError: Event loop is closed` / `attached to a different loop`.
- [ ] `pytest tests/unit/test_tasks.py -v` — зелёный (6+ тестов).
- [ ] Ветка `week-11` создана, коммит сделан.

> **Расхождение с ТЗ.** В задании сказано «`beat_schedule`: раз в сутки `clear_old_cache(days=30)`». **В референсном `celery_app.py` `beat_schedule` НЕ настроен.** Контейнер `beat` в compose есть, но без `beat_schedule` он ничего не запускает по расписанию. `clear_old_cache` вызывается только вручную (`from backend.worker.tasks import clear_old_cache; clear_old_cache.delay()`). Если хотите расписание — добавьте `beat_schedule` в `celery_app.conf` (пример в 11.5). В методичке — **по фактам кода**: `beat_schedule` нет, `beat` работает вхолостую.

---

## 11.0. Подготовка: ветка и проверка зависимостей

```powershell
cd D:\RUNG
git checkout -b week-11
```

Проверьте, что в `pyproject.toml` есть:

```toml
celery = "^5.6.3"
redis = "^8.1.0"
```

Проверьте, что `backend/worker/__init__.py` есть (создан в Этапе 2.1):

```powershell
dir backend\worker\
```

**Что вы должны увидеть** (`celery_app.py` и `tasks.py` пока отсутствуют):

```
    Directory: D:\RUNG\backend\worker

Mode    LastWriteTime    Length Name
----    -------------    ------ ----
-a---   ...                0 __init__.py
```

---

## 11.1. `backend/worker/celery_app.py` — приложение Celery

**Что это:** создаёт `celery_app` (singleton) с конфигом, подписывается на 2 shutdown-сигнала Celery для закрытия httpx-клиентов и event loop.

**Зачем это нужно:** Celery worker — отдельный процесс (`celery -A backend.worker.celery_app worker`). Задачи `index_translation_memory`, `batch_translate` и т.д. регистрируются в этом приложении. Без `celery_app` команда `celery` не найдёт задачи. Shutdown-сигналы — закрывают ресурсы при остановке контейнера.

Создайте `backend/worker/celery_app.py`:

```python
# backend/worker/celery_app.py
import logging

from celery import Celery
from celery.signals import worker_process_shutdown, worker_shutdown

from backend.core.config import get_settings
from backend.utils.http_client import close_ollama_clients

logger = logging.getLogger(__name__)
settings = get_settings()

celery_app = Celery(
    "rung",
    broker=settings.REDIS_URL,
    backend=settings.REDIS_URL,
    include=["backend.worker.tasks"],
)

celery_app.conf.update(
    task_serializer="json",
    accept_content=["json"],
    result_serializer="json",
    timezone="UTC",
    enable_utc=True,
    task_track_started=True,
    task_time_limit=30 * 60,
    task_soft_time_limit=25 * 60,
    worker_prefetch_multiplier=1,
    task_acks_late=True,
)

logger.info("Celery app configured with broker: %s", settings.REDIS_URL)


def _shutdown_worker_resources(**kwargs) -> None:
    """
    Release resources held by the process that actually ran tasks.

    Called on both ``worker_process_shutdown`` (fires in each prefork
    child) and ``worker_shutdown`` (fires in the main process, which is
    the executor under the solo pool). The handler is idempotent: if the
    process never created a worker loop, both branches are no-ops.

    Order matters. The httpx clients are bound to the worker's event
    loop, so they must be closed while the loop is still alive.
    ``close_worker_loop`` runs afterwards.
    """
    from backend.worker import tasks

    loop = tasks._worker_loop
    if loop is not None and not loop.is_closed():
        try:
            loop.run_until_complete(close_ollama_clients())
        except Exception as e:
            logger.warning("Failed to close httpx clients on shutdown: %s", e)

    try:
        tasks.close_worker_loop()
    except Exception as e:
        logger.warning("Failed to close worker event loop: %s", e)


worker_process_shutdown.connect(_shutdown_worker_resources)
worker_shutdown.connect(_shutdown_worker_resources)


if __name__ == "__main__":
    celery_app.start()
```

### Разбор ключевых решений

**Почему `Celery("rung", ...)` — имя `"rung"`.** Это имя app, используется для очередей в Redis (`rung` → `celery` ключ). Может быть любым, но фиксированным — если поменять, задачи в старой очереди потеряются.

**Почему `broker=settings.REDIS_URL` и `backend=settings.REDIS_URL` — один URL.** Redis используется и как broker (очередь задач), и как backend (хранение результатов). Один Redis-инстанс — два «канала». URL вида `redis://redis:6379/0` — база `0` для обоих. Можно было разделить (`/0` для broker, `/1` для backend), но для учебного проекта — один URL.

**Почему `include=["backend.worker.tasks"]` — строка, а не импорт.** Celery импортирует `backend.worker.tasks` при старте — модуль регистрирует задачи (`@celery_app.task`). Без `include` задачи не будут видны: worker запустится, но `celery_app.tasks` не будет содержать `index_translation_memory`. `include` — правильный способ, не прямой импорт (иначе циклическая зависимость: `tasks` импортирует `celery_app`, а `celery_app` — `tasks`).

**Почему `task_serializer="json"` и `accept_content=["json"]`.** Задачи сериализуются в JSON при отправке в Redis. `json` — безопаснее, чем `pickle` (не выполнит произвольный код при десериализации). `accept_content=["json"]` — worker принимает только JSON (не `pickle`), защита от подмены очереди.

**Почему `timezone="UTC"` + `enable_utc=True`.** Все `eta`/`countdown` в UTC. `datetime.now(timezone.utc)` в `clear_old_cache` — совпадает. Без `UTC` `now()` на сервере в TZ Europe/Moscow сдвинул бы `cutoff` на 3 часа — часть записей «не удалилась бы» или «удалилась бы лишняя».

**Почему `task_track_started=True`.** Задачи с длинным `run` (`index_translation_memory` на 5000 записей — 30+ с) показывают статус `STARTED` в `AsyncResult.state`. Без `True` состояние `PENDING` → `SUCCESS`, `STARTED` не виден. Полезно для отладки и мониторинга.

**Почему `task_time_limit=30 * 60` (30 минут) и `task_soft_time_limit=25 * 60` (25 минут).** Hard limit — 30 минут, после — SIGKILL. Soft — 25 минут, за 5 минут до — `SoftTimeLimitExceeded`, задача может поймать и сохранить прогресс. Для `batch_translate` на 1000 текстов × 10 с = 2.7 часа — 30 минут мало. Но 1000 текстов — экстремальный случай; 100 текстов × 10 с = 17 минут — влезает. Для увеличения — `task_time_limit=60 * 60 * 2` (2 часа), но тогда `worker` рискует зависнуть. Компромисс 30 минут.

**Почему `worker_prefetch_multiplier=1`.** Worker берёт из Redis ровно 1 задачу за раз (не «пачку из 4»). Дефолт 4 — worker забирает 4 задачи, выполняет по одной. Если одна долгая (`batch_translate` — 30 минут), три остальные ждут, а другие worker'ы их не получают (задача «занята»). `prefetch_multiplier=1` — fair distribution: одна задача на worker за раз. `--concurrency=1` (в compose) — 1 worker-процесс. Итого: 1 задача одновременно. Медленно, но предсказуемо.

**Почему `task_acks_late=True`.** ACK отправляется **после** выполнения задачи, а не при получении. Если worker упадёт во время выполнения — задача вернётся в очередь (будет перезапущена другим worker'ом или при рестарте). Без `acks_late` worker упал → задача «потеряна» (ACK уже отправлен, но результат не записан). Для `index_translation_memory` — критично (потеря = нет переиндексации).

**Почему `_shutdown_worker_resources` — `**kwargs`.** Celery signal-handlers вызываются с kwarg'ами (`sender`, `signal`, ...). Функция без `**kwargs` упала бы с `TypeError: unexpected keyword argument`. `**kwargs` — принимаем всё, игнорируем.

**Почему `from backend.worker import tasks` **внутри** функции, а не на module-level.** `celery_app.py` импортируется `backend.worker.tasks` (через `include`), а `tasks.py` импортирует `celery_app`. Module-level импорт `tasks` в `celery_app` создал бы цикл. Inline — разрывает.

**Почему `loop.run_until_complete(close_ollama_clients())` — синхронный вызов async-функции.** Shutdown-handler **не** async (Celery не поддерживает async signals). `loop` — persistent loop worker'а (создан в `tasks._get_worker_loop`). `run_until_complete` запускает корутину в этом loop'е — пока loop жив, httpx-клиенты закрываются корректно. Если бы `loop` уже был закрыт — `RuntimeError`, ловим `except Exception` с warning.

**Почему **сначала** `close_ollama_clients`, **потом** `close_worker_loop`.** httpx-клиенты привязаны к loop'у. Если закрыть loop первым, `await client.aclose()` упадёт с `RuntimeError: Event loop is closed`. Порядок: клиенты → loop. Комментарий в коде фиксирует.

**Почему `_shutdown_worker_resources` idempotent.** Вызывается дважды (в prefork child и в main). Если loop уже закрыт (первый вызов закрыл) — второй вызов видит `loop.is_closed()` → True, оба блока — no-op. Или `loop is None` (не создавался) — тоже no-op.

**Почему **два** сигнала — `worker_process_shutdown` и `worker_shutdown`.** В **prefork** pool (`--concurrency=2`): `worker_process_shutdown` срабатывает в каждом child-процессе (там живёт `_worker_loop`), `worker_shutdown` — в main-процессе (loop'а нет, no-op). В **solo** pool (`--concurrency=1`): `worker_shutdown` срабатывает в main-процессе, где loop и был создан. Подписка на оба — универсально для обоих пулов.

**Почему `celery_app.start()` в `if __name__ == "__main__"`.** `python -m backend.worker.celery_app` запускает CLI Celery. В Docker используется `celery -A backend.worker.celery_app worker` (не `python -m`), но `celery_app.start()` — эквивалент. Оставлено для удобства прямого запуска.

---

## 11.2. `backend/worker/tasks.py` — persistent loop и задачи

**Что это:** persistent event loop на процесс + 6 задач. Ключевое — `_get_worker_loop()`, который создаёт loop **один раз** на процесс, а не на каждую задачу.

**Зачем persistent loop:** `async_engine` (Этап 3) привязан к event loop'у. Если каждая Celery-задача создаёт `asyncio.new_event_loop()`, движок «застревает» в первом loop'е, и вторая задача падает с `RuntimeError: Task ... got Future ... attached to a different loop`. Один loop на процесс — все задачи используют один движок, один пул соединений.

Создайте `backend/worker/tasks.py`:

```python
# backend/worker/tasks.py
import asyncio
import logging
import threading
from datetime import datetime, timedelta, timezone
from typing import Any, Dict, List, Optional

from sqlalchemy import delete

from backend.core.models import GlossaryEntry, TMEntry
from backend.data.glossary import glossary_manager
from backend.data.translation_memory import tm_manager
from backend.db.models import TranslationCache
from backend.db.session import AsyncSessionLocal
from backend.eval.runner import Evaluator
from backend.services.pipeline import perform_translation
from backend.worker.celery_app import celery_app

logger = logging.getLogger(__name__)

_MAX_CONCURRENT_TRANSLATIONS = 4

_worker_loop: Optional[asyncio.AbstractEventLoop] = None
_worker_loop_lock = threading.Lock()


def _get_worker_loop() -> asyncio.AbstractEventLoop:
    global _worker_loop
    if _worker_loop is None or _worker_loop.is_closed():
        with _worker_loop_lock:
            if _worker_loop is None or _worker_loop.is_closed():
                _worker_loop = asyncio.new_event_loop()
                asyncio.set_event_loop(_worker_loop)
    return _worker_loop


def _run_async(coro):
    return _get_worker_loop().run_until_complete(coro)


def close_worker_loop() -> None:
    """
    Close the persistent worker event loop.

    Called from the ``worker_shutdown`` signal in ``celery_app``. Safe to
    call when the loop was never created.
    """
    global _worker_loop
    if _worker_loop is not None and not _worker_loop.is_closed():
        _worker_loop.close()
        logger.info("Worker event loop closed")
    _worker_loop = None


@celery_app.task(bind=True, max_retries=3)
def index_translation_memory(self, force: bool = False):
    try:
        logger.info("Starting TM indexing...")
        _run_async(tm_manager.load_all_into_cache())
        logger.info("TM indexing completed.")
        return {"status": "success", "message": "TM indexed"}
    except Exception as e:
        logger.error("TM indexing failed: %s", e, exc_info=True)
        self.retry(exc=e, countdown=60)
        raise


@celery_app.task(bind=True, max_retries=2)
def batch_translate(
    self,
    texts: List[str],
    source_lang: str,
    target_lang: str,
    eval_mode: str = "full",
    domain: str = "general",
    audience: str = "general",
) -> List[Dict[str, Any]]:
    async def _translate_all():
        semaphore = asyncio.Semaphore(_MAX_CONCURRENT_TRANSLATIONS)

        async def _bounded(text: str):
            async with semaphore:
                return await perform_translation(
                    text,
                    source_lang,
                    target_lang,
                    eval_mode=eval_mode,
                    domain=domain,
                    audience=audience,
                )

        tasks = [_bounded(text) for text in texts]
        return await asyncio.gather(*tasks)

    try:
        logger.info(
            "Batch translation started: %d texts, %s->%s, mode=%s, "
            "domain=%s, audience=%s",
            len(texts), source_lang, target_lang,
            eval_mode, domain, audience,
        )
        results = _run_async(_translate_all())
        logger.info("Batch translation completed: %d texts", len(results))
        return results
    except Exception as e:
        logger.error("Batch translation failed: %s", e, exc_info=True)
        self.retry(exc=e, countdown=30)
        raise


@celery_app.task
def run_evaluation_async(
    sources: List[str],
    hypotheses: List[str],
    references: List[str],
    lang: str = "en",
):
    try:
        evaluator = Evaluator()
        results = evaluator.evaluate(hypotheses, references, sources, lang=lang)
        logger.info("Evaluation completed: %s", results)
        return results
    except Exception as e:
        logger.error("Evaluation failed: %s", e, exc_info=True)
        raise


@celery_app.task
def clear_old_cache(days: int = 30):
    async def _clear():
        async with AsyncSessionLocal() as session:
            cutoff = datetime.now(timezone.utc) - timedelta(days=days)
            stmt = delete(TranslationCache).where(
                TranslationCache.created_at < cutoff
            )
            result = await session.execute(stmt)
            await session.commit()
            deleted = result.rowcount or 0
            logger.info(
                "Cleared %d old cache entries (older than %d days)",
                deleted, days,
            )
            return {"deleted": deleted}

    try:
        return _run_async(_clear())
    except Exception as e:
        logger.error("Failed to clear old cache: %s", e, exc_info=True)
        raise


@celery_app.task
def add_glossary_bulk(entries: List[Dict[str, str]]):
    async def _add():
        added = 0
        for entry_data in entries:
            entry = GlossaryEntry(**entry_data)
            await glossary_manager.add_entry(entry)
            added += 1
        logger.info("Added %d glossary entries", added)
        return {"added": added}

    try:
        return _run_async(_add())
    except Exception as e:
        logger.error("Bulk glossary import failed: %s", e, exc_info=True)
        raise


@celery_app.task
def add_tm_bulk(entries: List[Dict[str, str]]):
    async def _add():
        tm_entries = [TMEntry(**entry_data) for entry_data in entries]
        added = await tm_manager.add_many(tm_entries)
        logger.info("Added %d TM entries", added)
        return {"added": added}

    try:
        result = _run_async(_add())
    except Exception as e:
        logger.error("Bulk TM import failed: %s", e, exc_info=True)
        raise

    if result["added"] > 0:
        index_translation_memory.delay()
        logger.info(
            "Triggered TM reindex after inserting %d entries",
            result["added"],
        )

    return result
```

### Разбор ключевых решений

**Почему `_worker_loop: Optional[asyncio.AbstractEventLoop] = None` — module-level global.** Один loop на процесс. `global` в `_get_worker_loop` и `close_worker_loop` — общая переменная.

**Почему `_worker_loop_lock = threading.Lock()` — `threading`, не `asyncio`.** `_get_worker_loop` — синхронная функция, может вызываться из разных потоков (в prefork pool — из child-процессов, в solo — из main). `threading.Lock` — правильный примитив для синхронного кода. `asyncio.Lock` требует `await`, а `_get_worker_loop` sync.

**Почему двойная проверка `if _worker_loop is None or _worker_loop.is_closed()` — и до, и внутри lock'а.** Первая проверка — fast path (типичный случай — loop уже создан, не закрыт). Вторая — под lock'ом: два потока одновременно видят «loop closed», оба идут под lock, первый создаёт новый, второй видит «loop создан» и выходит. Классический double-checked locking.

**Почему `asyncio.set_event_loop(_worker_loop)`.** Регистрирует loop как «текущий» для потока. Некоторые библиотеки (`httpx` в `get_ollama_client`) вызывают `asyncio.get_running_loop()` — без `set_event_loop` они бы не нашли loop. `set_event_loop` — важно для совместимости.

**Почему `_run_async(coro)` = `_get_worker_loop().run_until_complete(coro)`.** Каждая Celery-задача sync (Celery не поддерживает async tasks из коробки). `_run_async` запускает корутину в persistent loop. `run_until_complete` — блокирует sync-вызов до завершения корутины.

**Почему `close_worker_loop` сбрасывает `_worker_loop = None` после close.** Чтобы следующий `_get_worker_loop` создал новый loop (если задача запустится после shutdown — теоретически возможно при graceful restart). Иначе `_worker_loop.is_closed()` → True, но `_worker_loop` не None — и `if` в `_get_worker_loop` корректно сработает (создаст новый). Всё равно `_worker_loop = None` — чище, явно «loop больше нет».

**Почему `logger.info("Worker event loop closed")` — в `close_worker_loop`, а не в `_shutdown_worker_resources`.** Логируется факт закрытия loop. Если в shutdown-логе видно это сообщение — значит, handler сработал. Критерий готовности проверяет именно эту строку.

**Почему `@celery_app.task(bind=True, max_retries=3)` для `index_translation_memory`.** `bind=True` — `self` доступен (для `self.retry`). `max_retries=3` — до 3 повторов при ошибке. `countdown=60` — через 60 секунд (Qdrant может быть временно перегружен). 3 попытки × 60 с = 3 минуты на восстановление.

**Почему `index_translation_memory(force: bool = False)` — параметр `force` не используется.** Задел на будущее: «полная переиндексация с drop collection vs только новые записи». Сейчас всегда `load_all_into_cache()` (Этап 7.3: `_rebuild_all_indexes` → drop + create + add). Параметр оставлен для совместимости с ТЗ.

**Почему `_run_async(tm_manager.load_all_into_cache())` — не `_run_async(tm_manager.ensure_loaded())`.** `load_all_into_cache` (Этап 7.3) — **полная** переиндексация, drop+create Qdrant. `ensure_loaded` — ленивая, только если `_loaded=False`. Для `index_translation_memory` — полная, потому что это «явная команда переиндексировать». В web-процессе `_ensure_indexes_loaded` — ленивая, без drop (Этап 7.3: «Does NOT recreate the Qdrant collection»).

**Почему `except Exception: self.retry(exc=e, countdown=60); raise`.** `self.retry` выбрасывает `Retry`-исключение — Celery планирует повтор. `raise` после — недостижим (Retry пробрасывается), но защищает от «забыли raise». Стандартный паттерн Celery.

**Почему `batch_translate(bind=True, max_retries=2)`, `countdown=30`.** Batch-перевод — тяжёлый (сотни текстов). 2 retry × 30 с — быстрее, чем index (там 60 с). Если 100 текстов, один упал на timeout — retry перезапустит **весь** batch (не только упавший). Поэтому `perform_translation` (через `_bounded`) сам ловит исключения — `safe_perform_translation` в `perform_translation`... хотя нет, `perform_translation` не safe. Смотрю код: `_bounded` вызывает `perform_translation`, который **бросает** при ошибке. Значит, один упавший текст → `gather` → exception → весь batch retry. `max_retries=2` — 2 полных перезапуска. Если один текст стабильно падает (например, невалидный язык) — 3 попытки, потом `raise`. Все 100 текстов будут перезапущены 3 раза. **Ограничение**: для частичных ошибок лучше `safe_perform_translation` (но в референсе — `perform_translation`).

**Почему `_MAX_CONCURRENT_TRANSLATIONS = 4` в `batch_translate`.** Семафор на 4. `_bounded` захватывает слот, `perform_translation`, отпускает. `asyncio.gather` запускает все `_bounded` одновременно, но реально в LLM идут 4. Совпадает с `OLLAMA_NUM_PARALLEL=2` × 2 (запас) и `_translation_semaphore` в `translate.py` (Этап 9). Общая пропускная способность к Ollama — ограничена.

**Почему `_translate_all` — вложенная async-функция.** `_run_async(coro)` принимает корутину. `_translate_all()` — фабрика корутины. Плюс семафор создаётся **внутри** корутины (а не на module-level) — каждый batch имеет свой семафор (нет утечки между задачами).

**Почему `run_evaluation_async` — sync (без `_run_async`).** `Evaluator.evaluate` (Этап 7) — sync (sacrebleu, evaluate — синхронные). `run_evaluation_async` не работает с async-БД. `Evaluator.evaluate` блокирует поток — но Celery-задача sync, это норма. `_run_async` не нужен.

**Почему `clear_old_cache(days: int = 30)` — sync-задача с async-внутренностью.** Как `index_translation_memory`: sync-обёртка, внутри `_run_async(_clear())`. `_clear` — async, работает с `AsyncSessionLocal` (Этап 3).

**Почему `cutoff = datetime.now(timezone.utc) - timedelta(days=days)`.** Записи старше `cutoff` удаляются. `timezone.utc` — совпадает с `server_default=func.now()` в `TranslationCache.created_at` (Postgres хранит `TIMESTAMPTZ` в UTC). `days=30` — дефолт. `result.rowcount` — количество удалённых строк.

**Почему `delete(TranslationCache).where(...)` — один запрос, а не `select` + `delete`.** `DELETE ... WHERE` — атомарно, БД сама находит строки. `SELECT` + `DELETE` — два round-trip, race condition. Один `DELETE` — правильно.

**Почему `add_glossary_bulk` — цикл по `glossary_manager.add_entry`, а не batch.** `glossary_manager.add_entry` не поддерживает batch (Этап 7: SELECT-then-INSERT, cache.clear на каждый). Для 1000 записей — 1000 SELECT + 1000 INSERT + 1000 cache.clear. Медленно, но работает. `add_tm_bulk` — через `add_many` (batch `INSERT ... ON CONFLICT`). `glossary_manager` — legacy, можно оптимизировать в будущем.

**Почему `add_glossary_bulk` не триггерит reindex.** Глоссарий не индексируется в Qdrant (только TM). BM25 — только для TM. Глоссарий — в БД, читается через `glossary_manager.get_entries` (ILIKE, не BM25). Переиндексация не нужна.

**Почему `add_tm_bulk` — `tm_manager.add_many` (batch).** `add_many` (Этап 7.3) — `INSERT ... ON CONFLICT DO NOTHING RETURNING id`, возвращает `len(inserted)`. Для 1000 записей — один INSERT, один commit. Быстро. `index_after_insert` (внутри `add_many`) обновляет `_tm_cache` и помечает пару в `_bm25_dirty` (ленивая пересборка — Этап 7.3).

**Почему `if result["added"] > 0: index_translation_memory.delay()`.** Если `added=0` (все дубликаты) — переиндексация не нужна. `delay()` — fire-and-forget, ставит задачу в очередь. `logger.info` после — для лога.

**Почему `add_tm_bulk` **не** вызывает `_run_async(tm_manager.index_after_insert(...))`.** `add_many` **внутри** уже вызывает `index_after_insert` (Этап 7.3). Дублировать не нужно. Но `index_after_insert` обновляет кэш **текущего** процесса (worker), а `index_translation_memory.delay()` — запускает **новую задачу**, которая пересоберёт BM25 из БД (worker'а). Двойная работа, но так в референсе.

**Почему `from backend.worker.celery_app import celery_app` — в начале.** `@celery_app.task` в момент импорта модуля регистрирует задачу. Без импорта `celery_app` задачи не будут видны. Циклическая зависимость: `celery_app` (include) импортирует `tasks`, `tasks` импортирует `celery_app` — Python обрабатывает через порядок: `celery_app` создаётся, потом `include` триггерит импорт `tasks`, `tasks` берёт уже созданный `celery_app`. Цикла нет.

**Почему `perform_translation` из `services.pipeline` — импортируется, хотя `batch_translate` его использует.** `perform_translation` (Этап 9.3) — центральная функция. `batch_translate` — пакетная обёртка. Хорошо: одна логика pipeline, не дублируется.

---

## 11.3. `infrastructure/docker-compose.yml` — worker и beat

**Что это:** добавить два сервиса в `docker-compose.yml` (после Этапа 8 там `db`, `redis`, `qdrant`, `ollama`, `ollama-init`, `app`; после Этапа 12 добавится `nginx`).

**Зачем это нужно:** без `worker` задачи в очереди Redis копятся, но никто их не выполняет. `beat` — для расписания (в референсе пустой, но контейнер есть).

Откройте `infrastructure/docker-compose.yml` и добавьте **перед `nginx`** (или после `app` — порядок не важен):

```yaml
  worker:
    build:
      context: ..
      dockerfile: infrastructure/Dockerfile
    healthcheck:
      disable: true
    env_file:
      - .env
    environment:
      PYTHONUNBUFFERED: 1
    deploy:
      resources:
        limits:
          memory: 10G
    stop_grace_period: 300s
    depends_on:
      db:
        condition: service_healthy
      redis:
        condition: service_healthy
      qdrant:
        condition: service_healthy
      ollama:
        condition: service_healthy
    restart: unless-stopped
    command:
      - celery
      - -A
      - backend.worker.celery_app
      - worker
      - --loglevel=info
      - --concurrency=1
    networks:
      - rung_internal

  beat:
    build:
      context: ..
      dockerfile: infrastructure/Dockerfile
    healthcheck:
      disable: true
    env_file:
      - .env
    environment:
      PYTHONUNBUFFERED: 1
    deploy:
      resources:
        limits:
          memory: 1G
    depends_on:
      db:
        condition: service_healthy
      redis:
        condition: service_healthy
      qdrant:
        condition: service_healthy
      ollama:
        condition: service_healthy
    restart: unless-stopped
    command:
      - celery
      - -A
      - backend.worker.celery_app
      - beat
      - --loglevel=info
    networks:
      - rung_internal
```

### Разбор ключевых решений

**Почему `healthcheck: disable: true`.** У `worker` и `beat` нет HTTP-эндпоинта. Проверять «жив ли» можно только через `celery inspect ping` (медленно, требует broker). Docker `healthcheck` без `test` не имеет смысла. `disable: true` — явно «не проверять». Оркестратор (Docker Compose) не будет помечать `unhealthy`, но при падении процесса `restart: unless-stopped` перезапустит.

**Почему `stop_grace_period: 300s` для worker.** `docker compose stop worker` отправляет SIGTERM, ждёт 300 с, потом SIGKILL. Celery worker по SIGTERM: перестаёт брать новые задачи, ждёт завершения текущей (до `task_soft_time_limit=25min`), потом shutdown. 300 с = 5 минут — хватает для большинства задач. Для `batch_translate` на 100 текстов (17 мин) — 300 с мало, SIGKILL прибьёт, задача потеряется (но `task_acks_late=True` вернёт в очередь). Компромисс: увеличить до 1800 с (30 мин) для прода; для учебного — 300.

**Почему `deploy.resources.limits.memory: 10G` для worker.** Worker использует `perform_translation` → embeddings, LLM-вызовы, Supervisor. Пиковая RAM — embeddings (~500 МБ) + контекст запроса (~100 МБ) + Python. 10G — с запасом. `beat` — только планировщик, 1G.

**Почему `--concurrency=1` для worker.** Один prefork-child. При `--concurrency=2` — два child-процесса, каждый со своим `_worker_loop` и `async_engine`. Ollama с `OLLAMA_NUM_PARALLEL=2` обрабатывает 2 одновременно. Компромисс: 1 worker = 1 задача одновременно (медленно, но предсказуемо). Для ускорения — `--concurrency=2` (2 задачи), но тогда `async_engine` в каждом child создаст свой пул (10+20=30 коннектов × 2 = 60 — близко к лимиту Postgres 100).

**Почему `depends_on: db/qdrant/ollama: service_healthy`, `redis: service_healthy`.** Worker использует БД (TM, glossary), Qdrant (индексация), Ollama (batch_translate), Redis (broker). Все 4 должны быть готовы. `service_healthy` ждёт healthcheck. Без — worker стартует раньше и падает на первом `await db.execute(...)`.

**Почему `beat` тоже зависит от всех 4 сервисов.** `beat` сам не обращается к БД/Qdrant/Ollama (только публикует задачи в Redis). Но `beat_schedule` (если бы был) — задачи, которые потом выполняются worker'ом. Формально `beat` нужен только `redis`. В референсе — все 4, «на всякий случай» (если `beat_schedule` добавит задачу, которая обращается к БД). Ограничение, но не ошибка.

**Почему `command: [celery, -A, backend.worker.celery_app, worker, --loglevel=info, --concurrency=1]`.** Развёрнутая форма (не shell). `celery -A <app> worker` — запуск worker'а. `-A backend.worker.celery_app` — приложение. `--loglevel=info` — INFO-логи (не DEBUG, чтобы не забивать). `--concurrency=1` — 1 child.

**Почему `build` как у `app` (`context: ..`, `dockerfile: infrastructure/Dockerfile`).** Тот же образ, что `app`. Worker запускает `celery`, не `uvicorn` — но код (backend/) и зависимости (celery, redis) — те же. Один Dockerfile, разные `command`. Экономия: не собирать второй образ.

**Почему `env_file: - .env`.** Worker читает `REDIS_URL`, `OLLAMA_HOST`, `DATABASE_URL` и т.д. — те же переменные, что `app`. `.env` — один для всех сервисов.

---

## 11.4. Связка HITL ↔ Celery (проверка)

В Этапе 10 в `active_learning.py` уже стоит:

```python
if inserted:
    from backend.worker.tasks import index_translation_memory
    index_translation_memory.delay()
```

До Этапа 11 это падало с `ImportError`, потому что `backend/worker/tasks.py` не существовал. Сейчас `tasks.py` создан — импорт работает. **Проверим связку end-to-end.**

### Запуск с worker

```powershell
cd D:\RUNG\infrastructure
docker compose up -d --build
```

**Что вы должны увидеть:**

```
[+] Running 8/8
 ✔ Container infrastructure-db-1       Healthy
 ✔ Container infrastructure-redis-1    Healthy
 ✔ Container infrastructure-qdrant-1   Healthy
 ✔ Container infrastructure-ollama-1   Healthy
 ✔ Container infrastructure-ollama-init-1 Exited
 ✔ Container infrastructure-app-1      Started
 ✔ Container infrastructure-worker-1   Started
 ✔ Container infrastructure-beat-1     Started
```

Проверьте статус:

```powershell
docker compose ps
```

**Что вы должны увидеть** (`worker` и `beat` в `running`, без healthcheck — `health: starting` или пусто):

```
NAME                          SERVICE       STATUS
infrastructure-db-1           db            Up (healthy)
infrastructure-redis-1        redis         Up (healthy)
infrastructure-qdrant-1       qdrant        Up (healthy)
infrastructure-ollama-1       ollama        Up (healthy)
infrastructure-ollama-init-1  ollama-init   Exited (0)
infrastructure-app-1          app           Up (healthy)
infrastructure-worker-1       worker        Up
infrastructure-beat-1         beat          Up
```

Проверьте лог worker'а:

```powershell
docker compose logs worker --tail=50
```

**Что вы должны увидеть** (фрагмент):

```
worker-1  |  -------------- celery@... v5.6.3
worker-1  | --- ***** -----
worker-1  | -- ******* ---- Linux-...
worker-1  | - *** --- * ---
worker-1  | - ** ---------- [config]
worker-1  | - ** ---------- .> app:         rung:0x...
worker-1  | - ** ---------- .> transport:   redis://redis:6379/0
worker-1  | - ** ---------- .> results:     redis://redis:6379/0
worker-1  | - *** --- * --- .> concurrency: 1 (prefork)
worker-1  | -- ******* ---- .> task events: OFF
worker-1  | --- ***** -----
worker-1  |  -------------- [queues]
worker-1  |                 .> celery           exchange=celery(direct) key=celery
worker-1  |
worker-1  | [tasks]
worker-1  |   . backend.worker.tasks.add_glossary_bulk
worker-1  |   . backend.worker.tasks.add_tm_bulk
worker-1  |   . backend.worker.tasks.batch_translate
worker-1  |   . backend.worker.tasks.clear_old_cache
worker-1  |   . backend.worker.tasks.index_translation_memory
worker-1  |   . backend.worker.tasks.run_evaluation_async
worker-1  | [2026-09-11 15:30:00,000: INFO/MainProcess] Connected to redis://redis:6379/0
worker-1  | [2026-09-11 15:30:00,000: INFO/MainProcess] celery@... ready.
```

`[tasks]` — 6 задач зарегистрированы. `celery@... ready` — worker готов. Если задач 0 — `include` в `celery_app.py` не сработал.

### Эксперимент: подтверждение через HITL → задача в worker'е

Создайте pending example (как в Этапе 10.5), подтвердите:

```powershell
$ADMIN_TOKEN = (curl -s -X POST http://localhost:8000/auth/login -H "Content-Type: application/json" -d "{\"email\":\"admin@rung.local\",\"password\":\"Admin1234\"}" | ConvertFrom-Json).access_token
$EXAMPLE_ID = "<id из /hitl/uncertain>"
curl -X POST "http://localhost:8000/hitl/uncertain/$EXAMPLE_ID/confirm" -H "Authorization: Bearer $ADMIN_TOKEN" -H "Content-Type: application/json" -d "{\"corrected_translation\": \"...\"}"
```

**Что вы должны увидеть** в ответе:

```json
{"status": "ok", "tm_inserted": true, "message": "Correction confirmed and added to TM"}
```

Проверьте лог worker'а **сразу**:

```powershell
docker compose logs worker --tail=20
```

**Что вы должны увидеть** (задача `index_translation_memory` получена):

```
worker-1  | [2026-09-11 15:31:00,000: INFO/MainProcess] Task backend.worker.tasks.index_translation_memory[...] received
worker-1  | [2026-09-11 15:31:00,000: INFO/ForkPoolWorker-1] Starting TM indexing...
worker-1  | [2026-09-11 15:32:00,000: INFO/ForkPoolWorker-1] Rebuilt all indexes for N entries
worker-1  | [2026-09-11 15:32:00,000: INFO/ForkPoolWorker-1] TM indexing completed.
worker-1  | [2026-09-11 15:32:00,000: INFO/ForkPoolWorker-1] Task backend.worker.tasks.index_translation_memory[...] succeeded in 60.0s: {'status': 'success', 'message': 'TM indexed'}
```

`received` → `Starting` → `succeeded`. Связка работает.

### Эксперимент: проверка, что новая TM-запись доступна

Сразу после `confirm`, в web-процессе:

```powershell
curl -X POST http://localhost:8000/translate -H "Authorization: Bearer $ADMIN_TOKEN" -H "Content-Type: application/json" -d "{\"text\": \"<исходный текст>\", \"source_lang\": \"en\", \"target_lang\": \"tt\"}"
```

**Что вы должны увидеть** в `tm_used`:

```json
{"tm_used": [{"id": "...", "en": "...", "tgt": "...", "source_lang": "en", "target_lang": "tt", "score": ...}], ...}
```

Запись из `confirm` есть в `tm_used` — `tm_manager.index_after_insert` (вызван в `confirm_correction`) обновил `_tm_cache` web-процесса. BM25 пересоберётся при следующем `search` (lazy `_bm25_dirty`).

### Эксперимент: shutdown worker

```powershell
docker compose stop worker
docker compose logs worker --tail=20
```

**Что вы должны увидеть** (graceful shutdown, `Worker event loop closed`):

```
worker-1  | [2026-09-11 15:35:00,000: INFO/MainProcess] worker: Warm shutdown (MainProcess)
worker-1  | [2026-09-11 15:35:00,000: INFO/MainProcess] Worker event loop closed
worker-1  | [2026-09-11 15:35:01,000: INFO/MainProcess] process ... exited with exitcode 0
```

**НЕ должно быть** `RuntimeError: Event loop is closed` или `attached to a different loop`. Если есть — `_shutdown_worker_resources` работает неправильно (порядок: клиенты → loop).

Верните:

```powershell
docker compose start worker
```

---

## 11.5. `beat_schedule` — расхождение с ТЗ

**В референсном `celery_app.py` `beat_schedule` НЕ настроен.** `beat` контейнер запускается (`celery beat`), но без `beat_schedule` не публикует никаких задач по расписанию. `clear_old_cache` вызывается только вручную:

```python
from backend.worker.tasks import clear_old_cache
clear_old_cache.delay(days=30)
```

**Если хотите расписание** (как в ТЗ) — добавьте в `celery_app.conf.update(...)`:

```python
celery_app.conf.beat_schedule = {
    "clear-old-cache-daily": {
        "task": "backend.worker.tasks.clear_old_cache",
        "schedule": 60 * 60 * 24,  # 24 hours
        "kwargs": {"days": 30},
    },
}
```

Это **отклонение от референсного кода**, но соответствует ТЗ. После добавления — `docker compose restart beat`, и в логах `beat` появится:

```
beat-1  | Scheduler: Sending due task clear-old-cache-daily (backend.worker.tasks.clear_old_cache)
```

И задача каждые 24 часа выполняется worker'ом.

**Решение.** Для сдачи по ТЗ — **добавьте** `beat_schedule`. Для 1:1 соответствия референсу — **не добавляйте**. В методичке по фактам: в референсе нет, в ТЗ есть. Студент выбирает.

---

## 11.6. Тесты

Тесты скинет пользователь. Здесь — что должно проверяться.

### `tests/unit/test_tasks.py`

**Настройка `task_always_eager=True`.** С `task_always_eager=True` `.delay()` выполняет задачу **синхронно**, не отправляя в Redis. Это позволяет тестировать логику задач без запуска worker'а.

```python
# tests/unit/test_tasks.py
import pytest
from unittest.mock import AsyncMock, patch

from backend.worker.celery_app import celery_app


@pytest.fixture(autouse=True)
def eager_mode():
    """Run tasks synchronously, no Redis needed."""
    celery_app.conf.task_always_eager = True
    celery_app.conf.task_eager_propagates = True
    yield
    celery_app.conf.task_always_eager = False
```

**Что тестируется:**

- `index_translation_memory()` — вызывает `tm_manager.load_all_into_cache()` (mock `tm_manager`, assert called), возвращает `{"status": "success", ...}`.
- `index_translation_memory()` при ошибке `load_all_into_cache` → `Retry`-исключение (Celery), лог `error`.
- `batch_translate(texts, "en", "ru")` — вызывает `perform_translation` для каждого text (mock, `AsyncMock`), возвращает список результатов.
- `batch_translate` с 5 текстами, семафор 4 → одновременно 4 `perform_translation` (проверить через mock с `side_effect` — считать одновременные вызовы).
- `run_evaluation_async(sources, hypotheses, references, "en")` — вызывает `Evaluator.evaluate`, возвращает метрики.
- `clear_old_cache(days=30)` — `_run_async(_clear())` вызывает `DELETE` с `cutoff` (mock `AsyncSessionLocal`).
- `add_glossary_bulk([{...}, {...}])` — вызывает `glossary_manager.add_entry` для каждой (mock), `{"added": N}`.
- `add_tm_bulk([{...}, {...}])` — вызывает `tm_manager.add_many` (mock, возвращает `len(inserted)`), `{"added": N}`.
- **`add_tm_bulk` при `added > 0`** — `index_translation_memory.delay()` вызван (mock `index_translation_memory.delay`).
- **`add_tm_bulk` при `added = 0`** — `index_translation_memory.delay()` **НЕ** вызван.

**Mock стратегия.**

```python
def test_add_tm_bulk_triggers_reindex(monkeypatch):
    async def mock_add_many(entries):
        return len(entries)
    monkeypatch.setattr(
        "backend.worker.tasks.tm_manager.add_many", mock_add_many
    )
    reindex_called = []
    monkeypatch.setattr(
        "backend.worker.tasks.index_translation_memory.delay",
        lambda: reindex_called.append(True),
    )
    result = add_tm_bulk.apply(args=[[
        {"en": "a", "tgt": "b", "source_lang": "en", "target_lang": "ru"},
    ]]).get()
    assert result["added"] == 1
    assert reindex_called == [True]


def test_add_tm_bulk_no_reindex_on_duplicate(monkeypatch):
    async def mock_add_many(entries):
        return 0  # all duplicates
    monkeypatch.setattr(
        "backend.worker.tasks.tm_manager.add_many", mock_add_many
    )
    reindex_called = []
    monkeypatch.setattr(
        "backend.worker.tasks.index_translation_memory.delay",
        lambda: reindex_called.append(True),
    )
    result = add_tm_bulk.apply(args=[[
        {"en": "a", "tgt": "b", "source_lang": "en", "target_lang": "ru"},
    ]]).get()
    assert result["added"] == 0
    assert reindex_called == []
```

**`.apply(args=[...])`** — синхронный вызов с аргументами. `task_always_eager=True` делает `.delay()` эквивалентным `.apply()`. `.apply()` — более явный, без зависимости от `task_always_eager`.

### `tests/unit/test_worker_loop.py`

- `_get_worker_loop()` — возвращает loop.
- `_get_worker_loop()` — повторный вызов возвращает **тот же** loop.
- После `close_worker_loop()` — `_get_worker_loop()` создаёт новый loop.
- `close_worker_loop()` идемпотентен — повторный вызов no-op.

```python
def test_persistent_loop():
    from backend.worker import tasks
    tasks.close_worker_loop()  # ensure clean state
    loop1 = tasks._get_worker_loop()
    loop2 = tasks._get_worker_loop()
    assert loop1 is loop2
    tasks.close_worker_loop()
    assert tasks._worker_loop is None
    loop3 = tasks._get_worker_loop()
    assert loop3 is not loop1
    tasks.close_worker_loop()
```

**Тонкость:** `_get_worker_loop` создаёт loop и регистрирует его как текущий. После теста — `close_worker_loop` в `finally`, иначе следующий тест получит чужой loop.

### `tests/unit/test_shutdown.py`

- `_shutdown_worker_resources()` при `_worker_loop = None` — no-op.
- `_shutdown_worker_resources()` при живом loop — вызывает `close_ollama_clients`, потом `close_worker_loop`.
- **Порядок**: mock `close_ollama_clients` и `close_worker_loop`, записать порядок вызовов.
- `_shutdown_worker_resources()` при уже закрытом loop — no-op для первой части, `close_worker_loop` — no-op.

**Фикстура `autouse` для `unload_hf_llm`** (Этап 8.9) — уже в `conftest.py`.

---

## 11.7. Финальная проверка

```powershell
python -m poetry run pytest tests/unit/test_tasks.py tests/unit/test_worker_loop.py tests/unit/test_shutdown.py -v
```

Все зелёные. Полный прогон:

```powershell
python -m poetry run pytest tests/ -v
```

Финальный чек-лист:

- [ ] `docker compose ps` — `worker`, `beat` в `Up`.
- [ ] `docker compose logs worker | grep "tasks"` — 6 задач зарегистрированы.
- [ ] `docker compose logs worker | grep "celery@.*ready"` — worker готов.
- [ ] `POST /hitl/uncertain/{id}/confirm` → `docker compose logs worker` показывает `Task backend.worker.tasks.index_translation_memory[...] succeeded`.
- [ ] `docker compose stop worker` → лог `Worker event loop closed`, нет `RuntimeError: Event loop is closed`.
- [ ] `docker compose start worker` — worker поднимается.
- [ ] `pytest tests/ -v` — зелёный.

Коммит и merge:

```powershell
git add -A
git commit -m "feat(week-11): add Celery worker, beat, 6 tasks, persistent loop"
git push -u origin week-11
```

---

## 11.8. Известные ограничения (не баги, отметить для отчёта)

1. **`beat_schedule` не настроен.** `beat` контейнер запускается, но ничего не публикует. `clear_old_cache` — только вручную. Если нужно — добавьте `beat_schedule` (см. 11.5).

2. **`batch_translate` не partial-fail-safe.** Если один текст в batch упал — `gather` бросит, весь batch в `Retry` (2 попытки), потом `raise`. 100 текстов × 3 попытки. Для partial-fail — используйте `safe_perform_translation` (есть в `services.pipeline`), но в референсе — `perform_translation`.

3. **`add_glossary_bulk` — медленный.** Цикл по `glossary_manager.add_entry` (SELECT-then-INSERT, cache.clear на каждый). Для 1000 записей — 1000 SELECT + 1000 INSERT. `add_tm_bulk` — быстрее (batch `INSERT ... ON CONFLICT`). `glossary_manager` можно оптимизировать (batch `INSERT ... ON CONFLICT`), но в референсе — цикл.

4. **`--concurrency=1`.** 1 задача одновременно. `index_translation_memory` (30+ с) блокирует `batch_translate` (17 мин). Для параллелизма — `--concurrency=2` (2 child-процесса, каждый со своим loop и пулом). Ограничение Postgres: 30 коннектов × 2 = 60, близко к лимиту 100.

5. **`index_translation_memory` не инкрементальный.** Всегда `load_all_into_cache()` → `_rebuild_all_indexes` → drop + create + add. Для 5000 записей — 30+ с. Инкрементальный (`index_after_insert`) уже сделан в `add_many`, но `index_translation_memory` — полный. Для больших БД нужен incremental-вариант.

6. **`clear_old_cache` не учитывает `hits`.** Удаляет по `created_at`, не по «популярности». Запись с `hits=1000`, но `created_at` 31 день назад — удалится. Для учебного — приемлемо. Для прода — `WHERE created_at < cutoff AND hits < threshold`.

---

## Что дальше

**Этап 12** — Streamlit (8 страниц), Docker + Nginx, eval-скрипты (`run_eval.py`, `paired_bootstrap.py`, `report_tables.py`, `check_tm_leakage.py`), `scripts/human_eval/`, README. После Этапа 12 — полный end-to-end: `docker compose up -d`, все сервисы в `healthy`, `/translate` работает, HITL пополняет TM, worker переиндексирует, Streamlit показывает UI.

**`backend/worker/tasks.py` больше не меняется** — все задачи (6) зарегистрированы, связки (`active_learning`, `feedback`, `add_tm_bulk`) работают. Этап 12 — только добавляет `nginx` в compose (Streamlit — отдельно, локально) и скрипты.